In [1]:
import Pkg

In [2]:
Pkg.activate(".")
Pkg.add("CpuId")
Pkg.add("ThreadPinning")
Pkg.add("BenchmarkTools")
Pkg.add("ProfileCanvas")
Pkg.add("QuantumControl")
Pkg.add("QuantumPropagators")
Pkg.add("QuantumControlTestUtils")
Pkg.instantiate()

  Activating project at `~/Documents/KrylovKitBenchmark`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/KrylovKitBenchmark/Manifest.t

In [3]:
using CpuId
cpuinfo()

| Cpu Property       | Value                                                      |
|:------------------ |:---------------------------------------------------------- |
| Brand              | Intel(R) Xeon(R) Gold 6226R CPU @ 2.90GHz                  |
| Vendor             | :Intel                                                     |
| Architecture       | :Skylake                                                   |
| Model              | Family: 0x06, Model: 0x55, Stepping: 0x07, Type: 0x00      |
| Cores              | 16 physical cores, 32 logical cores (on executing CPU)     |
|                    | Hyperthreading hardware capability detected                |
| Clock Frequencies  | 2900 / 3900 MHz (base/max), 100 MHz bus                    |
| Data Cache         | Level 1:3 : (32, 1024, 22528) kbytes                       |
|                    | 64 byte cache line size                                    |
| Address Size       | 48 bits virtual, 46 bits physical                          |
| SIMD               | 512 bit = 64 byte max. SIMD vector size                    |
| Time Stamp Counter | TSC is accessible via `rdtsc`                              |
|                    | TSC runs at constant rate (invariant from clock frequency) |
| Perf. Monitoring   | Performance Monitoring Counters (PMC) revision 4           |
|                    | Available hardware counters per logical core:              |
|                    | 3 fixed-function counters of 48 bit width                  |
|                    | 4 general-purpose counters of 48 bit width                 |
| Hypervisor         | No                                                         |


In [4]:
using ThreadPinning
pinthreads(:cores)
threadinfo()

Hostname: 	cuny
CPU(s): 	2 x Intel(R) Xeon(R) Gold 6226R CPU @ 2.90GHz
CPU target: 	cascadelake
Cores: 		32 (64 CPU-threads due to 2-way SMT)
NUMA domains: 	2 (16 cores each)

Julia threads: 	8

CPU socket 1
  0,32, 1,33, 2,34, 3,35, 4,36, 5,37, 6,38, 7,39, 
  8,40, 9,41, 10,42, 11,43, 12,44, 13,45, 14,46, 15,47

CPU socket 2
  16,48, 17,49, 18,50, 19,51, 20,52, 21,53, 22,54, 23,55, 
  24,56, 25,57, 26,58, 27,59, 28,60, 29,61, 30,62, 31,63


# = Julia thread, # = Julia thread on HT, # = >1 Julia thread

(Mapping: 1 => 0, 2 => 1, 3 => 2, 4 => 3, 5 => 4, ...)


In [5]:
filter(p -> contains(p[1], "THREAD"), ENV)

Dict{String, String} with 6 entries:
  "OPENBLAS_NUM_THREADS"   => "1"
  "VECLIB_MAXIMUM_THREADS" => "1"
  "OMP_NUM_THREADS"        => "1"
  "NUMEXPR_NUM_THREADS"    => "1"
  "MKL_NUM_THREADS"        => "1"
  "JULIA_NUM_THREADS"      => "8"

In [6]:
using QuantumControl: Trajectory, @threadsif
using QuantumPropagators: init_prop, propagate, Cheby
using QuantumControlTestUtils.DummyOptimization: dummy_control_problem
using ProfileCanvas

In [7]:
N = 100

100

In [8]:
problem = dummy_control_problem(; N, n_trajectories=Threads.nthreads(), n_steps=1000, density=1.0, hermitian=true)
tlist = problem.tlist;

In [9]:
function propagate_propagators(propagators; use_threads=true)
    numcoeffs_total = Threads.Atomic{Int64}(0)
    @threadsif use_threads for propagator in propagators
        propagate(propagator)
        Threads.atomic_add!(numcoeffs_total, propagator.wrk.n_coeffs)
    end
    return numcoeffs_total[]
end

propagate_propagators (generic function with 1 method)

In [10]:
function propagate_propagators(propagators; use_threads=true)
    @threadsif use_threads for propagator in propagators
        propagate(propagator)
    end
    return sum([p.wrk.n_coeffs for p in propagators])
end

propagate_propagators (generic function with 1 method)

In [11]:
propagators = [
    init_prop(traj.initial_state, traj.generator, problem.tlist; method=Cheby)
    for traj in problem.trajectories
]
propagate_propagators(propagators, use_threads=true)

112

In [12]:
using BenchmarkTools

In [13]:
bm_sequential = @benchmark propagate_propagators(propagators, use_threads=false) setup = (propagators = [
    init_prop(traj.initial_state, traj.generator, problem.tlist; method=Cheby)
    for traj in problem.trajectories
])

BenchmarkTools.Trial: 13 samples with 1 evaluation.
 Range (min … max):  384.493 ms … 399.566 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     390.913 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   391.324 ms ±   4.612 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▁  ▁         ▁▁     ▁  ▁ ▁ █             ▁      ▁   ▁       ▁  
  █▁▁█▁▁▁▁▁▁▁▁▁██▁▁▁▁▁█▁▁█▁█▁█▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁█▁▁▁█▁▁▁▁▁▁▁█ ▁
  384 ms           Histogram: frequency by time          400 ms <

 Memory estimate: 10.07 MiB, allocs estimate: 223714.

In [14]:
bm_parallel = @benchmark propagate_propagators(propagators, use_threads=true) setup = (propagators = [
    init_prop(traj.initial_state, traj.generator, problem.tlist; method=Cheby)
    for traj in problem.trajectories
])

BenchmarkTools.Trial: 44 samples with 1 evaluation.
 Range (min … max):   99.765 ms … 112.530 ms  ┊ GC (min … max): 0.00% … 9.00%
 Time  (median):     102.036 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   102.623 ms ±   2.382 ms  ┊ GC (mean ± σ):  1.08% ± 1.85%

    ▂     ▅▂ ▂█  ▂                                               
  ▅▁███▅███████▁▁█▅█▁▁▁▁▅▅▅▁▅▁▁▁▁▁█▁▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅ ▁
  99.8 ms          Histogram: frequency by time          113 ms <

 Memory estimate: 10.07 MiB, allocs estimate: 223758.

In [15]:
mean(bm_sequential.times) / mean(bm_parallel.times)

3.813203071906191

In [16]:
Base.GC.enable(false)
bm_parallel = @benchmark propagate_propagators(propagators, use_threads=true) setup = (propagators = [
    init_prop(traj.initial_state, traj.generator, problem.tlist; method=Cheby)
    for traj in problem.trajectories
])

BenchmarkTools.Trial: 42 samples with 1 evaluation.
 Range (min … max):  100.911 ms … 106.631 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     102.016 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   102.304 ms ±   1.263 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

   ▁ ▄  ▁▁  █▄▄▄▁ ▁▁   ▁      ▁                                  
  ▆█▆█▆▆██▆▆█████▁██▁▁▁█▁▁▁▁▁▁█▁▁▁▁▁▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆▁▁▁▆▁▁▁▁▁▁▆ ▁
  101 ms           Histogram: frequency by time          107 ms <

 Memory estimate: 10.07 MiB, allocs estimate: 223758.

In [17]:
Base.GC.enable(true);

In [18]:
mean(bm_sequential.times) / mean(bm_parallel.times)

3.8251026754814474

In [19]:
propagators = [
    init_prop(traj.initial_state, traj.generator, problem.tlist; method=Cheby)
    for traj in problem.trajectories
]
@profview propagate_propagators(propagators, use_threads=false)

ProfileCanvas.ProfileData(Dict{String, ProfileCanvas.ProfileFrame}("3" => ProfileCanvas.ProfileFrame("root", "", "", 0, 250, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("task_done_hook", "task.jl", "./task.jl", 694, 250, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("wait", "task.jl", "./task.jl", 1021, 250, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("poptask", "task.jl", "./task.jl", 1012, 250, missing, 0x10, missing, ProfileCanvas.ProfileFrame[])])])]), "4" => ProfileCanvas.ProfileFrame("root", "", "", 0, 250, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("task_done_hook", "task.jl", "./task.jl", 694, 250, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("wait", "task.jl", "./task.jl", 1021, 250, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("poptask", "task.jl", "./task.jl", 1012, 250, missing, 0x10, missing, ProfileCanvas.ProfileFrame[])])])]), "1" => ProfileCanvas.ProfileFrame("root", "", "", 0, 250, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#15", "eventloop.jl", "/home/goerz/.julia/packages/IJulia/bHdNn/src/eventloop.jl", 38, 247, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("eventloop", "eventloop.jl", "/home/goerz/.julia/packages/IJulia/bHdNn/src/eventloop.jl", 8, 247, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("invokelatest", "essentials.jl", "./essentials.jl", 1052, 247, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#invokelatest#2", "essentials.jl", "./essentials.jl", 1055, 247, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("execute_request", "execute_request.jl", "/home/goerz/.julia/packages/IJulia/bHdNn/src/execute_request.jl", 67, 247, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("softscope_include_string", "SoftGlobalScope.jl", "/home/goerz/.julia/packages/SoftGlobalScope/u4UzH/src/SoftGlobalScope.jl", 65, 247, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("include_string", "loading.jl", "./loading.jl", 2734, 247, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("eval", "boot.jl", "./boot.jl", 430, 247, missing, 0x01, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("propagate_propagators", "In[10]", "./In[10]", 1, 244, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#propagate_propagators#7", "In[10]", "./In[10]", 2, 244, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("macro expansion", "conditionalthreads.jl", "/home/goerz/.julia/packages/QuantumControl/sBK7k/src/conditionalthreads.jl", 36, 244, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("macro expansion", "In[10]", "./In[10]", 3, 244, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("propagate", "propagate.jl", "/home/goerz/.julia/packages/QuantumPropagators/0hpPp/src/propagate.jl", 283, 244, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#propagate#46", "propagate.jl", "/home/goerz/.julia/packages/QuantumPropagators/0hpPp/src/propagate.jl", 323, 244, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("prop_step!", "cheby_propagator.jl", "/home/goerz/.julia/packages/QuantumPropagators/0hpPp/src/cheby_propagator.jl", 349, 244, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("macro expansion", "TimerOutput.jl", "/home/goerz/.julia/packages/TimerOutputs/NRdsv/src/TimerOutput.jl", 253, 244, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("macro expansion", "cheby_propagator.jl", "/home/goerz/.julia/packages/QuantumPropagators/0hpPp/src/cheby_propagator.jl", 36